# 12 — Sensibilidades preespecificadas

Análisis técnico de horizontes, lookbacks y políticas de estancia sobre MIMIC-IV Demo v2.2. Solo se abren artefactos `development` y `validation`; el test permanece cerrado. Los subgrupos clínicos se posponen hasta disponer de tamaño suficiente y variables demográficas normalizadas.

In [ ]:
from pathlib import Path
import json, shutil, subprocess, sys, tempfile
import pandas as pd
from IPython.display import SVG, display
PROJECT_ROOT=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
sys.path.insert(0,str(PROJECT_ROOT/'src')) if str(PROJECT_ROOT/'src') not in sys.path else None
sys.path.insert(0,str(PROJECT_ROOT)) if str(PROJECT_ROOT) not in sys.path else None
from mimic_sepsis.artifacts import ArtifactStore, ArtifactValidationError
from mimic_sepsis.sensitivity import cohort_policy_summary, evaluate_horizon_lookback_grid, feature_coverage_grid
from scripts.build_demo_sofa_incremental import canonical_config
artifact_config=canonical_config(); model_config=json.loads((PROJECT_ROOT/'config/modeling.json').read_text()); sensitivity_config=json.loads((PROJECT_ROOT/'config/sensitivity.json').read_text())
valid=[]
for path in sorted((PROJECT_ROOT/'data/derived/sofa').glob('*/60_features/sepsis3_development_features.manifest.json')):
    try:
        manifest=ArtifactStore(path.parent).validate('sepsis3_development_features',expected_config=artifact_config); valid.append((manifest.created_at_utc,path.parents[1]))
    except (FileNotFoundError,ArtifactValidationError): pass
if not valid: raise RuntimeError('Ejecute primero los notebooks 00–11.')
RUN_ROOT=sorted(valid,key=lambda x:(x[0],str(x[1])))[-1][1]
landmark_store=ArtifactStore(RUN_ROOT/'50_landmarks'); feature_store=ArtifactStore(RUN_ROOT/'60_features')
print(f'Ejecución validada: {RUN_ROOT.name}')

## Sensibilidad estructural de la cohorte

In [ ]:
demo=PROJECT_ROOT/'data/mimic-iv-demo/2.2'
patients=pd.read_csv(demo/'hosp/patients.csv.gz'); admissions=pd.read_csv(demo/'hosp/admissions.csv.gz'); icustays=pd.read_csv(demo/'icu/icustays.csv.gz')
cohort_summary=cohort_policy_summary(patients,admissions,icustays,policies=sensitivity_config['cohort_policies'])
display(cohort_summary)
print('Esta tabla compara selección; no equivale a reconstruir fenotipo y modelos bajo cada política.')

## Conjuntos de riesgo por target y horizonte

In [ ]:
risk_rows=[]
for target in ('sepsis3','septic_shock'):
    for partition in ('development','validation'):
        frame=landmark_store.read_dataframe(f'{target}_{partition}_landmarks',expected_config=artifact_config)
        for horizon,group in frame.groupby('horizon_hours'):
            observed=group.horizon_observed
            risk_rows.append({'target':target,'partition':partition,'horizon_hours':int(horizon),'landmarks':len(group),'observed':int(observed.sum()),'censored':int((~observed).sum()),'positive':int(group.loc[observed,'outcome'].sum())})
risk_summary=pd.DataFrame(risk_rows); display(risk_summary)

## Cobertura por ventana predictora

In [ ]:
coverage=[]
for target in ('sepsis3','septic_shock'):
    for partition in ('development','validation'):
        frame=feature_store.read_dataframe(f'{target}_{partition}_features',expected_config=artifact_config)
        summary=feature_coverage_grid(frame,variables=artifact_config['features']['variables'],lookbacks_hours=sensitivity_config['lookbacks_hours']); summary['target']=target; summary['partition']=partition; coverage.append(summary)
coverage=pd.concat(coverage,ignore_index=True); display(coverage)

## Cuadrícula del baseline clínico Sepsis-3

In [ ]:
development_landmarks=landmark_store.read_dataframe('sepsis3_development_landmarks',expected_config=artifact_config)
validation_landmarks=landmark_store.read_dataframe('sepsis3_validation_landmarks',expected_config=artifact_config)
development_features=feature_store.read_dataframe('sepsis3_development_features',expected_config=artifact_config)
validation_features=feature_store.read_dataframe('sepsis3_validation_features',expected_config=artifact_config)
grid=evaluate_horizon_lookback_grid(development_landmarks,validation_landmarks,development_features,validation_features,primary_columns=model_config['clinical_baseline_features'],horizons_hours=sensitivity_config['horizons_hours'],lookbacks_hours=sensitivity_config['lookbacks_hours'],folds=model_config['cross_validation_folds'],c=model_config['logistic_c'],seed=model_config['seed'])
display(grid)
print('No seleccionar horizonte o ventana por estas métricas inestables del demo.')

## Visualización de sensibilidades con ggplot2

In [ ]:
rscript=shutil.which('Rscript')
if not rscript: raise RuntimeError('Rscript no está disponible.')
with tempfile.TemporaryDirectory() as tmp:
    tmp=Path(tmp); cohort_csv=tmp/'cohort.csv'; grid_csv=tmp/'grid.csv'; cohort_svg=tmp/'cohort.svg'; grid_svg=tmp/'grid.svg'; script=tmp/'plot.R'
    cohort_summary.to_csv(cohort_csv,index=False); grid.query("sample=='validation'").to_csv(grid_csv,index=False)
    script.write_text("""args <- commandArgs(trailingOnly=TRUE)
suppressPackageStartupMessages(library(ggplot2))
c <- read.csv(args[1]); g <- read.csv(args[2])
p1 <- ggplot(c,aes(policy,selected_icu_stays)) + geom_col(fill='#2878B5') + labs(title='Estancias seleccionadas por política',x=NULL,y='Estancias') + theme_minimal(base_size=11) + theme(axis.text.x=element_text(angle=20,hjust=1))
p2 <- ggplot(g,aes(factor(lookback_hours),factor(horizon_hours),fill=auprc)) + geom_tile() + geom_text(aes(label=sprintf('%.3f',auprc)),size=3) + scale_fill_viridis_c() + labs(title='AUPRC de validación — demo',subtitle='Descriptivo; no usar para seleccionar configuración',x='Lookback (h)',y='Horizonte (h)',fill='AUPRC') + theme_minimal(base_size=11)
ggsave(args[3],p1,width=7,height=4.8,device=grDevices::svg)
ggsave(args[4],p2,width=7,height=4.8,device=grDevices::svg)
""")
    result=subprocess.run([rscript,str(script),str(cohort_csv),str(grid_csv),str(cohort_svg),str(grid_svg)],capture_output=True,text=True)
    if result.returncode: raise RuntimeError(result.stderr)
    display(SVG(filename=str(cohort_svg)),SVG(filename=str(grid_svg)))

## Interpretación y pendientes

El análisis demuestra que las variantes se pueden ejecutar bajo el mismo contrato temporal y sin tocar test. El demo no permite escoger una variante por rendimiento. Antes de MIMIC-IV completo deben cerrarse la política primaria de cohorte, la cobertura SOFA, el proxy de shock y el plan de subgrupos; después se reconstruirá cada sensibilidad de extremo a extremo.